In [26]:
import pandas as pd

# 1. Load the raw dataset
# Drop any completely blank rows at the end of the file to ensure clean data
df = pd.read_csv('Project2_dataset.csv')
df = df.dropna(subset=['Departure Airport Name', 'Arrival Airport Name'])

In [27]:
corrections = {
    'Alberto Carnevalli Airport': 'Venezuela',
    'Charles M. Schulz Sonoma County Airport': 'United States',
    'Cheddi Jagan International Airport': 'Guyana',
    'Cibao International Airport': 'Dominican Republic',
    'Comodoro Arturo Merino Benitez International Airport': 'Chile', 
    'Eugene F. Correira International Airport': 'Guyana',
    'El Alto International Airport': 'Bolivia',
    'F. D. Roosevelt Airport': 'Netherlands Antilles',
    'Futuna Airport': 'Vanuatu',
    'General Jose Antonio Anzoategui International Airport': 'Venezuela',
    'JAGS McCartney International Airport': 'Turks and Caicos Islands',
    'London City Airport': 'United Kingdom',
    'London Gatwick Airport': 'United Kingdom',
    'London Heathrow Airport': 'United Kingdom',
    'London Luton Airport': 'United Kingdom',
    'London Stansted Airport': 'United Kingdom',
    'Luis Munoz Marin International Airport': 'Puerto Rico',
    'Mayor Buenaventura Vivas International Airport': 'Venezuela',
    'Norman Manley International Airport': 'Jamaica',
    'Norman Y. Mineta San Jose International Airport': 'United States',
    'Northwest Florida Beaches International Airport': 'United States',
    'Presidente Joao Batista Figueiredo Airport': 'Brazil',
    'St Petersburg Clearwater International Airport': 'United States',
    'St Pierre Airport': 'Saint Pierre and Miquelon',    
    'Sydney Kingsford Smith International Airport': 'Australia',
    
    'Albany Airport': 'Australia', 
    'Alexandria International Airport': 'United States',
    'Atlas Brasil Cantanhede Airport': 'Brazil',
    'Arturo Michelena International Airport': 'Venezuela',
    'Birmingham-Shuttlesworth International Airport': 'United States',
    'Cochin International Airport': 'India',
    'Florence Regional Airport': 'United States',
    'Fort Smith Regional Airport': 'United States',
    'Richmond Airport': 'Australia',
    'San Jose Airport': 'Philippines',
    'Santa Ana Airport': 'Solomon Islands',
    'Santa Fe Municipal Airport': 'United States',
    'Santa Rosa International Airport': 'Ecuador',
    'Tri-Cities Regional TN/VA Airport': 'United States',
    'Victoria Regional Airport': 'United States',
    'Waterloo Regional Airport': 'United States'
}

# 2. Loop through the dictionary and overwrite the country for both Departure and Arrival
for airport, correct_country in corrections.items():
    
    # Fix Departure Airports
    df.loc[
        df['Departure Airport Name'].str.startswith(airport, na=False), 
        'Departure Airport Country/Region'
    ] = correct_country
    
    # Fix Arrival Airports
    df.loc[
        df['Arrival Airport Name'].str.startswith(airport, na=False), 
        'Arrival Airport Country/Region'
    ] = correct_country

In [28]:
# ---------------------------------------------------------
# STEP 1: Extract Airport Nodes
# ---------------------------------------------------------
# We extract both departure and arrival airports, rename columns to match 
# our intended Neo4j properties, combine them, and drop duplicates.

dep_airports = df[['Departure Airport Name', 'Departure Airport City', 'Departure Airport Country/Region']].rename(columns={
    'Departure Airport Name': 'name',
    'Departure Airport City': 'city',
    'Departure Airport Country/Region': 'country'
})

arr_airports = df[['Arrival Airport Name', 'Arrival Airport City', 'Arrival Airport Country/Region']].rename(columns={
    'Arrival Airport Name': 'name',
    'Arrival Airport City': 'city',
    'Arrival Airport Country/Region': 'country'
})

airports_df = pd.concat([dep_airports, arr_airports]).drop_duplicates(subset=['name', 'city', 'country']).dropna(subset=['name'])

airports_df.to_csv('airports_nodes.csv', index=False)
print(f"Exported {len(airports_df)} unique airports to airports_nodes.csv")

Exported 2795 unique airports to airports_nodes.csv


In [29]:
# ---------------------------------------------------------
# STEP 2: Extract Airline Nodes
# ---------------------------------------------------------
# Even though we are storing airline names on the route, keeping a dedicated 
# Airline node CSV is best practice so you can easily list distinct Australian airlines.

airlines_df = df[['Airline Name', 'Airline Country']].rename(columns={
    'Airline Name': 'name',
    'Airline Country': 'country'
}).drop_duplicates(subset=['name']).dropna(subset=['name'])

airlines_df.to_csv('airlines_nodes.csv', index=False)
print(f"Exported {len(airlines_df)} unique airlines to airlines_nodes.csv")

Exported 488 unique airlines to airlines_nodes.csv


In [30]:
# ---------------------------------------------------------
# STEP 3: Extract & Aggregate Route Edges
# ---------------------------------------------------------
# We group by the departure and arrival airports, then aggregate the 
# airlines and planes into semicolon-separated lists.

# Custom function to handle the semicolon-separated Plane Name column
def get_unique_planes(plane_series):
    # Join all plane strings for this route with a semicolon
    all_planes_str = ';'.join(plane_series.dropna())
    # Split them back out to isolate each plane, get unique values, and rejoin
    unique_planes = list(set(all_planes_str.split(';')))
    # Filter out any empty strings that might have snuck in
    unique_planes = [p.strip() for p in unique_planes if p.strip()]
    return ';'.join(unique_planes)

# Custom function to get unique airlines
def get_unique_airlines(airline_series):
    unique_airlines = list(set(airline_series.dropna()))
    return ';'.join(unique_airlines)

# Group and aggregate
routes_df = df.groupby(['Departure Airport Name', 'Arrival Airport Name']).agg(
    airlines=('Airline Name', get_unique_airlines),
    aircraft_types=('Plane Name', get_unique_planes)
).reset_index()

# Rename to clean keys
routes_df.rename(columns={
    'Departure Airport Name': 'departure_airport',
    'Arrival Airport Name': 'arrival_airport'
}, inplace=True)

routes_df = routes_df[routes_df['departure_airport'] != routes_df['arrival_airport']].reset_index(drop=True)

routes_df.to_csv('routes_edges.csv', index=False)
print(f"Exported {len(routes_df)} unique aggregated routes to routes_edges.csv")


routes_df[routes_df["departure_airport"] == routes_df["arrival_airport"]]

Exported 32487 unique aggregated routes to routes_edges.csv


,departure_airport,arrival_airport,airlines,aircraft_types
